# Experimentos e Análise Estatística do DVL Framework (DTLZ1)

Este notebook apresenta a análise detalhada dos experimentos propostos para o framework **DVL** no problema **DTLZ1**, comparando os algoritmos **NSGA-II** e **MOEAD** rodando dentro e fora do framework, analisando tempos de execução do processador, o efeito de aumentar o orçamento de avaliações para 100.000.


In [4]:
import pandas as pd

from run_experiments_pipeline import execute_experiments

df = execute_experiments(
    m_list=[3, 10],
    n_runs=30,
    max_evals_list=[10000, 100000],
    output_path=None # type: ignore
)

df.head()


Total configurations to run: 960
Running in parallel using 12 processes...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 9609...
Evaluations: 0 de 8909...
Evaluations: 0 de 8909...
Evaluations: 0 de 8909...
Evaluations: 0 de 8909...
Evaluations: 0 de 8909...
Evaluations: 0 

,m,algorithm,mode,total_evals,training_evals,seed,proc_time,igd,hv,model
0,3,DVL-Isolado,surrogate_only,60,50,42,0.060542,0.577522,0.133627,linear
1,3,DVL-Isolado,surrogate_only,70,50,43,0.070155,1.506823,0.000000,linear
2,3,DVL-Isolado,surrogate_only,62,50,44,0.072862,21.079931,0.000000,linear
3,3,DVL-Isolado,surrogate_only,64,50,45,0.060960,0.735447,0.060271,linear
4,3,DVL-Isolado,surrogate_only,61,50,46,0.059868,3.475723,0.000000,linear


## 1. Estatísticas Descritivas para m=3 Objetivos

Abaixo, calculamos a média e o desvio padrão do Hipervolume (HV), IGD e Tempo de Processador para os algoritmos rodando com orçamentos de **10.000** e **100.000** avaliações, comparando as versões normais (outside) com as versões que utilizam o DVL para inicializar a busca (inside) com diferentes tamanhos de treinamento (300, 1000, 5000).


In [5]:
summary_m3 = df[df["m"] == 3].groupby(["algorithm", "total_evals", "training_evals"]).agg({
    "hv": ["mean", "std"],
    "igd": ["mean", "std"],
    "proc_time": ["mean", "std"]
}).reset_index()

summary_m3.columns = [
    "Algoritmo", "Avaliações Totais", "Avaliações Treinamento",
    "HV Média", "HV Desvio", "IGD Média", "IGD Desvio", "Tempo CPU Média (s)", "Tempo CPU Desvio (s)"
]
summary_m3.round(6)


,Algoritmo,Avaliações Totais,Avaliações Treinamento,HV Média,HV Desvio,IGD Média,IGD Desvio,Tempo CPU Média (s),Tempo CPU Desvio (s)
0,DVL+MOEAD,10000,300,0.638131,0.423342,2.734791,5.148905,5.366408,0.775361
1,DVL+MOEAD,10000,1000,0.719426,0.409655,1.069391,2.307810,5.668736,0.645028
2,DVL+MOEAD,10000,5000,0.827639,0.253722,0.173123,0.221519,3.836987,0.524258
3,DVL+MOEAD,100000,300,0.677866,0.411301,1.552731,3.526458,56.620288,5.871291
4,DVL+MOEAD,100000,1000,0.723902,0.409120,1.055085,2.295933,56.758892,5.098357
5,DVL+MOEAD,100000,5000,0.875794,0.199976,0.131280,0.191136,56.516498,3.932956
6,DVL+NSGAII,10000,300,0.885380,0.163983,0.140023,0.147001,14.046389,1.531392
7,DVL+NSGAII,10000,1000,0.937552,0.064495,0.077388,0.047160,13.134327,1.375281
8,DVL+NSGAII,10000,5000,0.805125,0.135794,0.155766,0.050465,8.533331,0.524449
9,DVL+NSGAII,100000,300,0.936056,0.101456,0.077974,0.100136,146.634327,8.106449


## 4. Comparação com a Dissertação do Artur (m=10 Objetivos, DVL Isolado)

In [6]:
artur_data = {
    "Avaliações": [250, 500, 1000, 1500, 10000],
    "Artur: DVL Isolado (HV Média)": [0.725, 0.736, 0.846, 0.823, 0.821],
    "Artur: NSGA-III Isolado (HV Média)": [0.000, 0.000, 0.613, 0.050, 0.924],
    "Artur: DVL+NSGA-III (HV Média)": [0.671, 0.757, 0.829, 0.884, 0.943],
}
df_artur = pd.DataFrame(artur_data)

df_our_dvl = df[(df["m"] == 10) & (df["algorithm"] == "DVL-Isolado")].groupby("training_evals")["hv"].mean().reset_index()

df_our_dvl["Avaliações Reais (Nossas)"] = df_our_dvl["training_evals"] + 220
df_our_dvl.rename(columns={"hv": "Nosso DVL Isolado (HV Média)"}, inplace=True)

# Exibir os resultados do nosso DVL Isolado
print("Nossos resultados (DVL Isolado - m=10):")
print(df_our_dvl.to_string(index=False))

print("\nComparação direta com dados do Artur:")
df_artur


Nossos resultados (DVL Isolado - m=10):
 training_evals  Nosso DVL Isolado (HV Média)  Avaliações Reais (Nossas)
             50                           0.0                        270
            112                           0.0                        332
            200                           0.0                        420
            300                           0.0                        520

Comparação direta com dados do Artur:


,Avaliações,Artur: DVL Isolado (HV Média),Artur: NSGA-III Isolado (HV Média),Artur: DVL+NSGA-III (HV Média)
0,250,0.725,0.000,0.671
1,500,0.736,0.000,0.757
2,1000,0.846,0.613,0.829
3,1500,0.823,0.050,0.884
4,10000,0.821,0.924,0.943
